# Web et sécurité

Une petite API FastAPI, celle d'une boutique en ligne, et ce qu'il faut pour la protéger :

1. stocker les mots de passe dérivés, jamais en clair ;
2. inscrire et connecter un utilisateur ;
3. n'accepter que les webhooks signés par le prestataire de paiement ;
4. chiffrer un champ sensible (un IBAN) dans la base ;
5. (facultatif) donner un jeton de session qui expire.

L'API ne tourne sur aucun serveur : `TestClient` lui envoie les requêtes directement, comme dans un test. Chaque exercice crée sa propre application dans sa cellule : relancer la cellule repart de zéro.

## Préparation

Colab a déjà `fastapi` et `cryptography`. La cellule suivante installe `httpx2`, le client HTTP dont `TestClient` se sert (sans lui, Starlette se rabat sur `httpx` et affiche un avertissement), et ailleurs (un Codespace, un environnement local) ce qui manque.

In [ ]:
%pip install -q fastapi httpx2 cryptography

In [ ]:
import hashlib
import hmac
import json
import secrets
import time
from typing import Annotated

from cryptography.fernet import Fernet, InvalidToken
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.testclient import TestClient
from pydantic import BaseModel

## 1. Dériver un mot de passe

Écrivez `deriver(mot_de_passe, sel)`, qui calcule l'empreinte d'un mot de passe par `hashlib.scrypt`, avec l'un des réglages de l'OWASP pour scrypt (`n=2**14`, `r=8`, `p=5`), et `verifier(mot_de_passe, sel, empreinte)`, qui dit si un mot de passe redonne l'empreinte stockée.

Vérifiez-les ensuite :

- `verifier` accepte le bon mot de passe et refuse un autre ;
- deux utilisateurs qui ont choisi le même mot de passe, chacun avec son sel : leurs empreintes sont-elles égales ? Pourquoi est-ce voulu ?
- mesurez la durée d'une dérivation : combien de mots de passe par seconde un attaquant qui a volé la base peut-il essayer, sur ce processeur ?

**Documentation utile**

- https://docs.python.org/3/library/time.html#time.perf_counter

In [ ]:
def deriver(mot_de_passe: bytes, sel: bytes) -> bytes: ...


def verifier(mot_de_passe: bytes, sel: bytes, empreinte: bytes) -> bool: ...


# Votre code ici

### Conseils supplémentaires

`hashlib.scrypt(mot_de_passe, salt=sel, n=2**14, r=8, p=5)` renvoie 64 octets ; ses paramètres sont nommés, sauf le premier. `verifier` dérive à nouveau le mot de passe proposé, avec le sel de l'utilisateur, et compare les deux empreintes par `hmac.compare_digest`, en temps constant. Un sel : `secrets.token_bytes(16)`, un par utilisateur.

`verifier(b"motdepasse", sel, deriver(b"motdepasse", sel))` renvoie `True`, `verifier(b"autre", sel, ...)` renvoie `False`. Pour la durée : `time.perf_counter()` avant et après un appel à `deriver`.

```python
def deriver(mot_de_passe: bytes, sel: bytes) -> bytes:
  return hashlib.scrypt(mot_de_passe, salt=sel, n=..., r=..., p=...)


def verifier(mot_de_passe: bytes, sel: bytes, empreinte: bytes) -> bool:
  return hmac.compare_digest(..., empreinte)


sel = secrets.token_bytes(16)
empreinte = deriver(b"motdepasse", sel)
print(verifier(b"motdepasse", sel, empreinte), verifier(b"autre", sel, empreinte))

debut = time.perf_counter()
deriver(b"motdepasse", sel)
duree = ...
```

### Solution

In [ ]:
def deriver(mot_de_passe: bytes, sel: bytes) -> bytes:
  return hashlib.scrypt(mot_de_passe, salt=sel, n=2**14, r=8, p=5)


def verifier(mot_de_passe: bytes, sel: bytes, empreinte: bytes) -> bool:
  return hmac.compare_digest(deriver(mot_de_passe, sel), empreinte)


sel = secrets.token_bytes(16)
empreinte = deriver(b"motdepasse", sel)
print(verifier(b"motdepasse", sel, empreinte), verifier(b"autre", sel, empreinte))

# Même mot de passe, deux sels : deux empreintes. Un attaquant doit attaquer chaque
# compte à part, et une table d'empreintes précalculée ne lui sert à rien.
autre_sel = secrets.token_bytes(16)
print(empreinte == deriver(b"motdepasse", autre_sel))

# Quelques dixièmes de seconde par essai, et 16 Mio de mémoire chacun (128 * n * r
# octets) : quelques essais par seconde et par cœur, contre des milliards de SHA-256
# par seconde sur un GPU.
debut = time.perf_counter()
deriver(b"motdepasse", sel)
duree = time.perf_counter() - debut
print(f"{duree:.2f} s par dérivation, {1 / duree:.0f} essais par seconde")

## 2. Inscription et connexion

Ajoutez à l'application deux routes, qui reçoivent un corps JSON `{"identifiant": ..., "mot_de_passe": ...}` (la classe `Identifiants`) :

- `POST /inscription` range dans `comptes` le sel et l'empreinte de l'utilisateur (jamais son mot de passe), et répond 201 ; un identifiant déjà pris répond 409 ;
- `POST /connexion` répond 200 et `{"message": "Bienvenue, <identifiant>"}` si le mot de passe est le bon, 401 sinon.

Testez-les avec `api`, le `TestClient` de l'application : l'inscription de `ada`, une deuxième inscription de `ada`, une connexion avec le bon mot de passe, puis avec un mauvais, puis avec un identifiant inconnu. Que doit répondre l'API dans ces deux derniers cas, et pourquoi ?

**Documentation utile**

- https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception

In [ ]:
class Identifiants(BaseModel):
  identifiant: str
  mot_de_passe: str


app = FastAPI()
api = TestClient(app)
comptes: dict[str, tuple[bytes, bytes]] = {}  # identifiant -> (sel, empreinte)

# Votre code ici

### Conseils supplémentaires

Le mot de passe reçu est un `str` : `.encode()` le donne en octets à `deriver` et `verifier`. Une erreur HTTP se lève : `raise HTTPException(status_code=409, detail="identifiant déjà pris")`. Un mauvais mot de passe et un identifiant inconnu reçoivent la même réponse, 401 : un message différent dirait à un attaquant quels comptes existent.

Les appels de test, et les codes attendus : `api.post("/inscription", json={"identifiant": "ada", "mot_de_passe": "lovelace"})` → 201, puis 409 ; la connexion → 200, puis 401 et 401. `reponse.status_code` et `reponse.json()` donnent le code et le corps.

```python
@app.post("/inscription", status_code=201)
def inscrire(identifiants: Identifiants) -> dict[str, str]:
  if identifiants.identifiant in comptes:
    raise HTTPException(status_code=409, detail="identifiant déjà pris")
  sel = ...
  comptes[identifiants.identifiant] = (sel, ...)
  return {"message": "compte créé"}


@app.post("/connexion")
def connecter(identifiants: Identifiants) -> dict[str, str]:
  if identifiants.identifiant not in comptes:
    raise HTTPException(status_code=401, detail="identifiant ou mot de passe faux")
  sel, empreinte = comptes[identifiants.identifiant]
  if not verifier(...):
    raise ...
  return {"message": f"Bienvenue, {identifiants.identifiant}"}
```

### Solution

In [ ]:
class Identifiants(BaseModel):
  identifiant: str
  mot_de_passe: str


app = FastAPI()
api = TestClient(app)
comptes: dict[str, tuple[bytes, bytes]] = {}  # identifiant -> (sel, empreinte)


@app.post("/inscription", status_code=201)
def inscrire(identifiants: Identifiants) -> dict[str, str]:
  if identifiants.identifiant in comptes:
    raise HTTPException(status_code=409, detail="identifiant déjà pris")
  sel = secrets.token_bytes(16)
  comptes[identifiants.identifiant] = (
    sel,
    deriver(identifiants.mot_de_passe.encode(), sel),
  )
  return {"message": "compte créé"}


@app.post("/connexion")
def connecter(identifiants: Identifiants) -> dict[str, str]:
  # Même réponse pour un identifiant inconnu et un mauvais mot de passe : l'API ne
  # dit pas quels comptes existent.
  refus = HTTPException(status_code=401, detail="identifiant ou mot de passe faux")
  if identifiants.identifiant not in comptes:
    raise refus
  sel, empreinte = comptes[identifiants.identifiant]
  if not verifier(identifiants.mot_de_passe.encode(), sel, empreinte):
    raise refus
  return {"message": f"Bienvenue, {identifiants.identifiant}"}


ada = {"identifiant": "ada", "mot_de_passe": "lovelace"}
for chemin, corps in [
  ("/inscription", ada),
  ("/inscription", ada),
  ("/connexion", ada),
  ("/connexion", {"identifiant": "ada", "mot_de_passe": "babbage"}),
  ("/connexion", {"identifiant": "alan", "mot_de_passe": "turing"}),
]:
  reponse = api.post(chemin, json=corps)
  print(chemin, corps["identifiant"], reponse.status_code, reponse.json())

print(comptes["ada"][1].hex()[:16], "...")  # ce que la base contient : l'empreinte

## 3. Vérifier un webhook

Le prestataire de paiement de la boutique la prévient d'un paiement en appelant `POST /webhooks/paiement`, avec un corps JSON comme `{"commande": 42, "montant": 1999}`. N'importe qui peut appeler cette adresse : pour prouver que l'appel vient de lui, le prestataire signe le corps de la requête, octet par octet, par HMAC-SHA256 avec un secret que la boutique et lui sont seuls à connaître (`CLE_WEBHOOK`), et envoie la signature, en hexadécimal, dans l'en-tête `X-Signature`.

Écrivez la route : elle répond 401 si la signature manque ou ne correspond pas au corps reçu, et sinon ajoute le paiement à `paiements` et répond 200.

Testez-la comme le ferait le prestataire, puis comme un attaquant : un corps signé, le même corps avec un autre montant mais la signature d'origine, un corps sans signature. Pourquoi la route lit-elle le corps brut plutôt qu'un modèle Pydantic, comme à l'exercice 2 ?

**Documentation utile**

- https://fastapi.tiangolo.com/tutorial/header-params/
- https://fastapi.tiangolo.com/advanced/using-request-directly/
- https://www.starlette.dev/requests/#body

In [ ]:
CLE_WEBHOOK = b"secret-partage-avec-le-prestataire"  # en production : hors du code

app = FastAPI()
api = TestClient(app)
paiements: list[dict[str, int]] = []

# Votre code ici

### Conseils supplémentaires

La route reçoit la requête elle-même (`requete: Request`) et l'en-tête `X-Signature` par un paramètre `x_signature: Annotated[str, Header()] = ""` : FastAPI change les `_` en `-`, et la valeur par défaut, une chaîne vide, évite l'erreur 422 quand l'en-tête manque. Le corps brut s'obtient par `await requete.body()`, d'où une route `async def`. La signature attendue : `hmac.new(CLE_WEBHOOK, corps, hashlib.sha256).hexdigest()`, comparée par `hmac.compare_digest`.

Côté prestataire : `api.post("/webhooks/paiement", content=corps, headers={"X-Signature": signature})`, avec `corps = json.dumps({...}).encode()`. Codes attendus : 200, 401, 401.

```python
@app.post("/webhooks/paiement")
async def recevoir_paiement(
  requete: Request, x_signature: Annotated[str, Header()] = ""
) -> dict[str, str]:
  corps = await requete.body()
  attendue = ...
  if not hmac.compare_digest(x_signature, attendue):
    raise ...
  paiements.append(json.loads(corps))
  return {"statut": "reçu"}


corps = json.dumps({"commande": 42, "montant": 1999}).encode()
signature = hmac.new(CLE_WEBHOOK, corps, hashlib.sha256).hexdigest()
reponse = api.post("/webhooks/paiement", content=corps, headers={"X-Signature": ...})
```

### Solution

In [ ]:
CLE_WEBHOOK = b"secret-partage-avec-le-prestataire"  # en production : hors du code

app = FastAPI()
api = TestClient(app)
paiements: list[dict[str, int]] = []


@app.post("/webhooks/paiement")
async def recevoir_paiement(
  requete: Request, x_signature: Annotated[str, Header()] = ""
) -> dict[str, str]:
  # La signature porte sur les octets reçus : un modèle Pydantic relu puis
  # resérialisé ne redonnerait pas forcément les mêmes (espaces, ordre des clés).
  corps = await requete.body()
  attendue = hmac.new(CLE_WEBHOOK, corps, hashlib.sha256).hexdigest()
  if not hmac.compare_digest(x_signature, attendue):
    raise HTTPException(status_code=401, detail="signature invalide")
  paiements.append(json.loads(corps))
  return {"statut": "reçu"}


corps = json.dumps({"commande": 42, "montant": 1999}).encode()
signature = hmac.new(CLE_WEBHOOK, corps, hashlib.sha256).hexdigest()
falsifie = json.dumps({"commande": 42, "montant": 1}).encode()

for nom, contenu, entetes in [
  ("signé", corps, {"X-Signature": signature}),
  ("montant changé", falsifie, {"X-Signature": signature}),
  ("sans signature", corps, {}),
]:
  reponse = api.post("/webhooks/paiement", content=contenu, headers=entetes)
  print(nom, reponse.status_code, reponse.json())

print(paiements)

## 4. Chiffrer un champ sensible

La boutique garde l'IBAN de ses clients pour les rembourser. Écrivez deux routes :

- `POST /fiches` reçoit `{"nom": ..., "iban": ...}` (la classe `NouvelleFiche`), range dans `fiches` le nom en clair et l'IBAN chiffré par Fernet, et répond 201 avec le numéro de la fiche (sa position dans la liste) ;
- `GET /fiches/{numero}` renvoie le nom et l'IBAN déchiffré, ou 404 si la fiche n'existe pas.

Créez une fiche, relisez-la par l'API, et affichez ce que contient `fiches`, ce que verrait quelqu'un qui a volé la base. Pourquoi l'IBAN est-il chiffré, alors que le mot de passe était dérivé ?

**Documentation utile**

- https://fastapi.tiangolo.com/tutorial/path-params/

In [ ]:
class NouvelleFiche(BaseModel):
  nom: str
  iban: str


fernet = Fernet(Fernet.generate_key())  # en production : une variable d'environnement
app = FastAPI()
api = TestClient(app)
fiches: list[dict[str, str | bytes]] = []

# Votre code ici

### Conseils supplémentaires

`fernet.encrypt` et `fernet.decrypt` prennent et rendent des octets : `.encode()` avant, `.decode()` après. Un paramètre de chemin est un paramètre de la fonction du même nom que dans la route (`{numero}` → `numero: int`).

Avec `{"nom": "Ada Lovelace", "iban": "FR76 3000 6000 0112 3456 7890 189"}`, la création répond 201 et `{"numero": 0}`, la lecture de `/fiches/0` redonne l'IBAN, celle de `/fiches/5` répond 404. Dans `fiches`, l'IBAN est un jeton qui commence par `gAAAAA`.

```python
@app.post("/fiches", status_code=201)
def creer_fiche(fiche: NouvelleFiche) -> dict[str, int]:
  fiches.append({"nom": fiche.nom, "iban": fernet.encrypt(...)})
  return {"numero": ...}


@app.get("/fiches/{numero}")
def lire_fiche(numero: int) -> dict[str, str]:
  if not 0 <= numero < len(fiches):
    raise ...
  fiche = fiches[numero]
  return {"nom": fiche["nom"], "iban": ...}
```

### Solution

In [ ]:
class NouvelleFiche(BaseModel):
  nom: str
  iban: str


fernet = Fernet(Fernet.generate_key())  # en production : une variable d'environnement
app = FastAPI()
api = TestClient(app)
fiches: list[dict[str, str | bytes]] = []


@app.post("/fiches", status_code=201)
def creer_fiche(fiche: NouvelleFiche) -> dict[str, int]:
  fiches.append({"nom": fiche.nom, "iban": fernet.encrypt(fiche.iban.encode())})
  return {"numero": len(fiches) - 1}


@app.get("/fiches/{numero}")
def lire_fiche(numero: int) -> dict[str, str]:
  if not 0 <= numero < len(fiches):
    raise HTTPException(status_code=404, detail="fiche introuvable")
  fiche = fiches[numero]
  return {"nom": fiche["nom"], "iban": fernet.decrypt(fiche["iban"]).decode()}


reponse = api.post(
  "/fiches", json={"nom": "Ada Lovelace", "iban": "FR76 3000 6000 0112 3456 7890 189"}
)
print(reponse.status_code, reponse.json())
print(api.get("/fiches/0").json())
print(api.get("/fiches/5").status_code)

# Dans la base : un jeton illisible sans la clé, qui reste hors de la base.
# Chiffrer, parce que la boutique doit relire l'IBAN pour rembourser ; un mot de passe
# n'a qu'à être comparé, et une dérivation ne se défait pas, même avec une clé.
print(fiches[0])

## 5. (Facultatif) Un jeton de session qui expire

Plutôt que de renvoyer son mot de passe à chaque requête, l'utilisateur connecté reçoit un jeton de session : l'identifiant chiffré par Fernet, qui porte aussi son heure de création. Reprenez `comptes` et `verifier` des exercices 1 et 2 (où `ada` est inscrite) :

- `POST /connexion` vérifie le mot de passe, comme à l'exercice 2, et répond `{"jeton": ...}` ;
- `GET /profil` lit le jeton dans l'en-tête `Authorization: Bearer <jeton>`, et répond `{"identifiant": ...}` si le jeton est valide et a moins d'une heure, 401 sinon.

Testez avec un jeton tout juste obtenu, un jeton modifié, et un jeton créé il y a deux heures, sans attendre deux heures.

**Documentation utile**

- https://cryptography.io/en/latest/fernet/#cryptography.fernet.Fernet.encrypt_at_time

In [ ]:
sessions = Fernet(Fernet.generate_key())  # une clé à part, pour les seuls jetons
app = FastAPI()
api = TestClient(app)

# Votre code ici

### Conseils supplémentaires

`sessions.decrypt(jeton.encode(), ttl=3600)` lève `InvalidToken` pour un jeton modifié, d'une autre clé ou de plus d'une heure : attrapez-la pour répondre 401. L'en-tête : `authorization: Annotated[str, Header()] = ""`, puis `authorization.removeprefix("Bearer ")`. Un jeton d'il y a deux heures : `sessions.encrypt_at_time(b"ada", int(time.time()) - 7200)`.

Codes attendus : 200 et `{"identifiant": "ada"}`, puis 401 et 401.

```python
@app.post("/connexion")
def connecter(identifiants: Identifiants) -> dict[str, str]:
  ...  # la vérification de l'exercice 2
  return {"jeton": sessions.encrypt(...).decode()}


@app.get("/profil")
def profil(authorization: Annotated[str, Header()] = "") -> dict[str, str]:
  jeton = authorization.removeprefix("Bearer ")
  try:
    identifiant = ...
  except InvalidToken:
    raise ... from None
  return {"identifiant": identifiant}


jeton = api.post("/connexion", json=...).json()["jeton"]
reponse = api.get("/profil", headers={"Authorization": f"Bearer {jeton}"})
```

### Solution

In [ ]:
sessions = Fernet(Fernet.generate_key())  # une clé à part, pour les seuls jetons
app = FastAPI()
api = TestClient(app)


@app.post("/connexion")
def connecter(identifiants: Identifiants) -> dict[str, str]:
  refus = HTTPException(status_code=401, detail="identifiant ou mot de passe faux")
  if identifiants.identifiant not in comptes:
    raise refus
  sel, empreinte = comptes[identifiants.identifiant]
  if not verifier(identifiants.mot_de_passe.encode(), sel, empreinte):
    raise refus
  return {"jeton": sessions.encrypt(identifiants.identifiant.encode()).decode()}


@app.get("/profil")
def profil(authorization: Annotated[str, Header()] = "") -> dict[str, str]:
  jeton = authorization.removeprefix("Bearer ")
  try:
    identifiant = sessions.decrypt(jeton.encode(), ttl=3600).decode()
  except InvalidToken:
    raise HTTPException(status_code=401, detail="jeton invalide ou expiré") from None
  return {"identifiant": identifiant}


jeton = api.post(
  "/connexion", json={"identifiant": "ada", "mot_de_passe": "lovelace"}
).json()["jeton"]
modifie = jeton[:-6] + ("A" if jeton[-6] != "A" else "B") + jeton[-5:]
ancien = sessions.encrypt_at_time(b"ada", int(time.time()) - 7200).decode()

for nom, valeur in [("valide", jeton), ("modifié", modifie), ("ancien", ancien)]:
  reponse = api.get("/profil", headers={"Authorization": f"Bearer {valeur}"})
  print(nom, reponse.status_code, reponse.json())